# Socratic AI Tutor — Production Fine-Tuning Pipeline

## Model: Qwen3-0.5B → Q4_K_M GGUF (~350 MB)

**Goal:** Fine-tune a 0.5B model that handles three behaviors:
1. **Socratic tutoring** — guides with questions, uses `<think>` blocks (DS/ML concepts)
2. **Code assistance** — provides direct code examples (when user asks for code)
3. **Casual conversation** — handles greetings and chitchat naturally

**Key fixes over previous version:**

| Issue | Previous | Fixed |
|---|---|---|
| Think blocks | 0% generated at inference | Explicit `<think>` in ALL training samples + generation enforcement |
| Response types | Socratic only | 3-mode: Socratic + Code + Casual |
| Data size | 234 conversations | 299 (234 original + 65 new: code, greetings, diverse Socratic) |
| System prompt | Vague | Unified multi-behavior prompt matching production |
| MAX_LENGTH | 512 (too short for code) | 768 (covers code responses) |
| Quantization | Not done in notebook | Full GGUF Q4_K_M export included |
| Evaluation | Only Socratic metrics | Per-category evaluation (Socratic, Code, Casual) |

In [1]:
%pip install -q torch datasets transformers peft bitsandbytes scikit-learn huggingface_hub accelerate trl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 40.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 528.8/528.8 kB 47.8 MB/s eta 0:00:00


In [2]:
# ============================================================================
# Cell 1: Imports, Seed, and System Prompt
# ============================================================================

import json, random, re, gc, os
from typing import Dict, List, Any, Tuple
from copy import deepcopy

import numpy as np
import torch
from datasets import Dataset
from peft import LoraConfig, get_peft_model, TaskType
from transformers import (
    AutoTokenizer, AutoModelForCausalLM,
    TrainingArguments, Trainer, EarlyStoppingCallback
)
from dataclasses import dataclass

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True

# ============================================================================
# UNIFIED SYSTEM PROMPT
# This single prompt teaches the model three behaviors:
# 1. Socratic tutoring (conceptual questions)
# 2. Direct code help (implementation requests)
# 3. Casual conversation (greetings, chitchat)
#
# The model learns to detect intent from the user message and route to the
# correct behavior. This matches the production system prompt.
# ============================================================================
SYSTEM_PROMPT = """You are a Socratic AI tutor specializing in data science and machine learning.

RULES:
1. ALWAYS begin your response with a thinking block containing your reasoning.
2. For conceptual questions: respond with ONE guiding question. NEVER give direct answers. If the student is stuck, give a small hint before your question.
3. For code questions: guide the student to write the code themselves through Socratic questioning.
4. For casual messages (greetings, thanks, chitchat): respond warmly and naturally.

Always start with a thinking block. This is mandatory."""

print("\u2713 Environment configured")
print(f"  - PyTorch: {torch.__version__}")
print(f"  - CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"  - GPU: {torch.cuda.get_device_name(0)}")
    print(f"  - VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")


✓ Environment configured
  - PyTorch: 2.10.0+cu128
  - CUDA: True
  - GPU: NVIDIA L4
  - VRAM: 23.7 GB


In [3]:
from huggingface_hub import login
login()
print("\u2713 Authenticated")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


✓ Authenticated


In [4]:
# ============================================================================
# Cell 2: Load Model + Tokenizer
# ============================================================================

MODEL_NAME = "Qwen/Qwen3-0.6B"

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME, trust_remote_code=True, padding_side="right"
)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.pad_token_id = tokenizer.eos_token_id

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    trust_remote_code=True,
    attn_implementation="eager"
)
model.config.use_cache = False

print(f"\u2713 Model loaded: {model.num_parameters():,} params")

config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

...

model.safetensors:   0%|          | 0.00/1.50G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.embed_tokens.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

✓ Model loaded: 751,632,384 params


In [5]:
# ============================================================================
# Cell 3: Load and Merge Training Data
# ============================================================================
# We merge the original Socratic data with supplementary data covering:
# - Code Q&A examples (direct code help)
# - Casual greetings/chitchat
# - Additional diverse Socratic conversations
# ============================================================================

SOCRATIC_PATH = "multi_socratic_data.json"
SUPPLEMENTARY_PATH = "supplementary_data.json"

with open(SOCRATIC_PATH, 'r') as f:
    socratic_data = json.load(f)

with open(SUPPLEMENTARY_PATH, 'r') as f:
    supplementary_data = json.load(f)

print(f"Original Socratic conversations: {len(socratic_data)}")
print(f"Supplementary conversations: {len(supplementary_data)}")

# Count categories in supplementary
cats = {}
for s in supplementary_data:
    c = s.get('category', 'unknown')
    cats[c] = cats.get(c, 0) + 1
print(f"  Breakdown: {cats}")

# Strip the 'category' field before merging (not needed for training)
clean_supplementary = []
for conv in supplementary_data:
    clean_supplementary.append({"messages": conv["messages"]})

# Merge
all_data = socratic_data + clean_supplementary
print(f"\nTotal merged conversations: {len(all_data)}")

Original Socratic conversations: 234
Supplementary conversations: 73
  Breakdown: {'greeting': 17, 'code': 34, 'socratic': 22}

Total merged conversations: 307


In [6]:
# ============================================================================
# Cell 4: Data Validation and Normalization
# ============================================================================
# Ensure every assistant response has a <think> block.
# This is CRITICAL — the previous model had 0% think block generation
# because ~1.6% of training samples were missing think blocks, and the
# model learned that skipping think was acceptable.
# ============================================================================

def validate_and_fix_conversation(conv: Dict, idx: int) -> Dict:
    """Ensure every assistant message has a <think> block."""
    messages = conv["messages"]
    fixed = []
    fixes = 0

    for msg in messages:
        if msg["role"] == "assistant":
            content = msg["content"]
            if "<think>" not in content:
                # Add a minimal think block
                content = "<think>Processing the student's message.</think>" + content
                fixes += 1
            fixed.append({"role": "assistant", "content": content})
        else:
            fixed.append(msg)

    if fixes > 0:
        print(f"  Conv {idx}: Fixed {fixes} assistant messages missing <think> blocks")

    return {"messages": fixed}

# Validate all conversations
print("Validating training data...")
validated_data = []
total_fixes = 0
total_assistant_msgs = 0

for i, conv in enumerate(all_data):
    n_assistant = sum(1 for m in conv["messages"] if m["role"] == "assistant")
    n_missing = sum(1 for m in conv["messages"]
                    if m["role"] == "assistant" and "<think>" not in m["content"])
    total_assistant_msgs += n_assistant
    total_fixes += n_missing
    validated_data.append(validate_and_fix_conversation(conv, i))

print(f"\n\u2713 Validation complete")
print(f"  Total assistant messages: {total_assistant_msgs}")
print(f"  Fixed (added <think>): {total_fixes}")
print(f"  Think block coverage: {(total_assistant_msgs - total_fixes) / total_assistant_msgs * 100:.1f}% → 100%")

Validating training data...
  Conv 155: Fixed 5 assistant messages missing <think> blocks
  Conv 156: Fixed 5 assistant messages missing <think> blocks
  Conv 157: Fixed 4 assistant messages missing <think> blocks

✓ Validation complete
  Total assistant messages: 991
  Fixed (added <think>): 14
  Think block coverage: 98.6% → 100%


In [7]:
# ============================================================================
# Cell 5: Data Augmentation via Conversation Windowing
# ============================================================================
# Each multi-turn conversation generates multiple training samples by taking
# progressively longer prefixes. This multiplies data ~3-4x without
# synthetic generation.
# ============================================================================

def augment_by_windowing(conversations: List[Dict]) -> List[Dict]:
    """Generate all valid sub-conversations ending on an assistant turn."""
    augmented = []
    for conv in conversations:
        messages = conv["messages"]

        # Separate system prompt
        if messages[0]["role"] == "system":
            dialogue = messages[1:]
        else:
            dialogue = messages

        # Generate windows ending on assistant turns
        for end_idx in range(2, len(dialogue) + 1, 2):
            window = dialogue[:end_idx]
            if window and window[-1]["role"] == "assistant":
                augmented.append({"messages": window})

    return augmented

augmented_data = augment_by_windowing(validated_data)
random.shuffle(augmented_data)

print(f"After windowing: {len(augmented_data)} samples")
print(f"Multiplier: {len(augmented_data) / len(validated_data):.1f}x")

After windowing: 991 samples
Multiplier: 3.2x


In [8]:
# ============================================================================
# Cell 6: Tokenization with Label Masking
# ============================================================================
# IMPORTANT: We set MAX_LENGTH = 768 to accommodate code responses which are
# longer than Socratic responses. 512 was too short and truncated code examples.
#
# Label masking ensures we ONLY train on assistant tokens — the model never
# learns to generate system prompts or user messages.
# ============================================================================

MAX_LENGTH = 768

def format_conversation(messages: List[Dict]) -> List[Dict]:
    """Prepend the unified system prompt to every conversation."""
    non_system = [m for m in messages if m["role"] != "system"]
    return [{"role": "system", "content": SYSTEM_PROMPT}] + non_system


def find_assistant_spans(input_ids: List[int]) -> List[tuple]:
    """Find token spans of assistant responses for label masking."""
    spans = []
    start_tag_id = tokenizer.convert_tokens_to_ids("<|im_start|>")
    end_tag_id = tokenizer.convert_tokens_to_ids("<|im_end|>")
    assistant_id = tokenizer.convert_tokens_to_ids("assistant")

    i = 0
    while i < len(input_ids):
        if (input_ids[i] == start_tag_id and
                i + 1 < len(input_ids) and
                input_ids[i + 1] == assistant_id):
            start_idx = i + 3  # skip <|im_start|>, 'assistant', '\n'
            end_idx = -1
            for j in range(start_idx, len(input_ids)):
                if input_ids[j] == end_tag_id:
                    end_idx = j
                    break
            if end_idx != -1:
                spans.append((start_idx, end_idx + 1))
                i = end_idx + 1
            else:
                spans.append((start_idx, len(input_ids)))
                break
        else:
            i += 1
    return spans


def tokenize_with_labels(examples: Dict) -> Dict:
    all_input_ids, all_attention_masks, all_labels = [], [], []

    for messages in examples["messages"]:
        formatted = format_conversation(messages)
        text = tokenizer.apply_chat_template(
            formatted, tokenize=False, add_generation_prompt=False
        )
        tok = tokenizer(
            text, truncation=True, max_length=MAX_LENGTH,
            padding="max_length", return_tensors=None
        )
        input_ids = tok["input_ids"]
        labels = [-100] * len(input_ids)
        for start, end in find_assistant_spans(input_ids):
            for idx in range(start, min(end, len(labels))):
                labels[idx] = input_ids[idx]

        all_input_ids.append(input_ids)
        all_attention_masks.append(tok["attention_mask"])
        all_labels.append(labels)

    return {
        "input_ids": all_input_ids,
        "attention_mask": all_attention_masks,
        "labels": all_labels
    }


dataset = Dataset.from_list(augmented_data)
tokenized = dataset.map(
    tokenize_with_labels, batched=True, batch_size=50,
    remove_columns=dataset.column_names, desc="Tokenizing"
)
split = tokenized.train_test_split(test_size=0.1, seed=SEED)
train_dataset = split["train"]
eval_dataset = split["test"]

print(f"\u2713 Train: {len(train_dataset):,} | Eval: {len(eval_dataset):,}")

# Verify label masking
sample = train_dataset[0]
n_trainable = sum(1 for t in sample['labels'] if t != -100)
n_total = sum(1 for t in sample['attention_mask'] if t == 1)
print(f"Label check: {n_trainable}/{n_total} real tokens are trainable")
assert n_trainable > 0, "ERROR: No trainable tokens! Label masking is broken."

# Verify <think> is in the trainable tokens
think_token_ids = tokenizer.encode("<think>", add_special_tokens=False)
sample_ids = sample['input_ids']
sample_labels = sample['labels']
think_found_in_labels = False
for i in range(len(sample_ids) - len(think_token_ids) + 1):
    if sample_ids[i:i+len(think_token_ids)] == think_token_ids:
        if sample_labels[i] != -100:
            think_found_in_labels = True
            break
print(f"<think> in trainable region: {think_found_in_labels}")

Tokenizing:   0%|          | 0/991 [00:00<?, ? examples/s]

✓ Train: 891 | Eval: 100
Label check: 229/442 real tokens are trainable
<think> in trainable region: True


In [9]:
# ============================================================================
# Cell 7: LoRA Configuration
# ============================================================================
# r=32 with RSLoRA gives good capacity for behavioral fine-tuning.
# Targeting all attention + MLP projections maximizes expressiveness.
# ============================================================================

model.gradient_checkpointing_enable()

lora_config = LoraConfig(
    r=32,
    lora_alpha=32,
    use_rslora=True,         # Rank-Stabilized LoRA — effective scale = alpha/sqrt(r)
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj"
    ],
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 20,185,088 || all params: 771,817,472 || trainable%: 2.6153


In [10]:
# ============================================================================
# Cell 8: Training Configuration
# ============================================================================
# Key decisions:
# - lr=2e-4: Standard for LoRA (2e-5 was too low in previous version)
# - Effective batch 16: Stable gradients
# - 3 epochs: Enough with ~1000 augmented samples, avoids overfit
# - Early stopping patience=5: Don't stop too early
# - weight_decay=0.01: Light regularization for LoRA
# ============================================================================

torch.cuda.empty_cache()
gc.collect()

BATCH_SIZE = 4
GRAD_ACCUM = 4
EPOCHS = 4
LR = 2e-4

total_steps = (len(train_dataset) // (BATCH_SIZE * GRAD_ACCUM)) * EPOCHS
warmup_steps = max(20, int(total_steps * 0.06))

print(f"Effective batch size: {BATCH_SIZE * GRAD_ACCUM}")
print(f"Total steps: {total_steps} | Warmup: {warmup_steps}")

training_args = TrainingArguments(
    output_dir="./qwen-socratic-lora-v3",

    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,

    num_train_epochs=EPOCHS,

    learning_rate=LR,
    lr_scheduler_type="cosine",
    warmup_steps=warmup_steps,

    bf16=True,
    optim="adamw_torch_fused",
    weight_decay=0.01,
    max_grad_norm=1.0,

    gradient_checkpointing=True,

    logging_steps=10,
    eval_strategy="steps",
    eval_steps=50,
    save_strategy="steps",
    save_steps=50,
    save_total_limit=3,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,

    remove_unused_columns=False,
    report_to="none",
    seed=SEED,
    dataloader_pin_memory=True,
)


@dataclass
class SocraticDataCollator:
    tokenizer: Any
    def __call__(self, features: List[Dict[str, Any]]) -> Dict[str, torch.Tensor]:
        return {
            "input_ids": torch.tensor([f["input_ids"] for f in features], dtype=torch.long),
            "attention_mask": torch.tensor([f["attention_mask"] for f in features], dtype=torch.long),
            "labels": torch.tensor([f["labels"] for f in features], dtype=torch.long),
        }


trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    data_collator=SocraticDataCollator(tokenizer=tokenizer),
    callbacks=[
        EarlyStoppingCallback(early_stopping_patience=5, early_stopping_threshold=0.005)
    ]
)

print("\u2713 Trainer initialized")

Effective batch size: 16
Total steps: 220 | Warmup: 20
✓ Trainer initialized


In [11]:
# ============================================================================
# Cell 9: Training
# ============================================================================

print("=" * 60)
print("STARTING TRAINING")
print("=" * 60)

train_result = trainer.train()

print(f"\n\u2713 Training complete")
print(f"  Runtime:     {train_result.metrics['train_runtime']:.1f}s")
print(f"  Final loss:  {train_result.metrics['train_loss']:.4f}")
print(f"  Steps:       {train_result.global_step}")

ADAPTER_DIR = "./socratic-qwen3-lora-v3-adapter"
model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)
print(f"\u2713 Adapter saved to {ADAPTER_DIR}")

STARTING TRAINING


Step,Training Loss,Validation Loss
50,1.571388,1.494723
100,0.768555,1.040800
150,0.336997,0.902143
200,0.162493,0.905103



✓ Training complete
  Runtime:     1200.9s
  Final loss:  0.8364
  Steps:       224
✓ Adapter saved to ./socratic-qwen3-lora-v3-adapter


In [12]:
# ============================================================================
# Cell 10: Merge LoRA into Base Model
# ============================================================================

merged_model = model.merge_and_unload()

MERGED_DIR = "./socratic-qwen3-merged-v3"
merged_model.save_pretrained(MERGED_DIR)
tokenizer.save_pretrained(MERGED_DIR)
print(f"\u2713 Merged model saved to {MERGED_DIR}")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

✓ Merged model saved to ./socratic-qwen3-merged-v3


In [ ]:
# ============================================================================
# Cell 11: Comprehensive Evaluation
# ============================================================================
# FIXES:
# 1. Added enable_thinking=True to apply_chat_template — this makes Qwen3
#    start generation with <think>\n, matching the training data format.
#    Without it, the model has to "decide" to emit <think> on its own,
#    which only worked 28.6% of the time.
#
# 2. Updated code evaluation to check for Socratic guidance (questions)
#    instead of code blocks — matches the training data where code questions
#    are answered with guiding questions, not direct code dumps.
# ============================================================================

def generate_response(question: str, model_to_use, max_tokens: int = 512) -> str:
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": question}
    ]

    prompt = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True,
        enable_thinking=True  # FIX: triggers <think> prefix in generation
    )

    inputs = tokenizer(prompt, return_tensors="pt").to(model_to_use.device)
    model_to_use.config.use_cache = True

    with torch.no_grad():
        outputs = model_to_use.generate(
            **inputs,
            max_new_tokens=max_tokens,
            do_sample=True,
            temperature=0.6,
            top_p=0.9,
            top_k=30,
            repetition_penalty=1.05,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    model_to_use.config.use_cache = False
    response = tokenizer.decode(
        outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True
    )
    return response.strip()


def evaluate_socratic(response: str) -> Dict[str, bool]:
    has_think = "<think>" in response and "</think>" in response
    post_think = response.split("</think>")[-1].strip() if "</think>" in response else response
    return {
        "Has Think Block": has_think,
        "Ends with Question": post_think.rstrip().endswith("?"),
        "No Direct Answer": not any(p in post_think.lower() for p in [
            "the answer is", "here's how", "you should", "the solution"
        ]),
        "Concise": len(post_think.split()) < 80
    }


def evaluate_code(response: str) -> Dict[str, bool]:
    """Evaluate code questions — model should guide with Socratic questions,
    NOT dump code directly."""
    has_think = "<think>" in response and "</think>" in response
    post_think = response.split("</think>")[-1].strip() if "</think>" in response else response
    return {
        "Has Think Block": has_think,
        "Guides with Question": post_think.rstrip().endswith("?"),
        "No Direct Code Dump": "```" not in post_think,
    }


def evaluate_casual(response: str) -> Dict[str, bool]:
    has_think = "<think>" in response and "</think>" in response
    post_think = response.split("</think>")[-1].strip() if "</think>" in response else response
    return {
        "Has Think Block": has_think,
        "Warm Tone": any(w in post_think.lower() for w in [
            "hello", "hey", "hi", "welcome", "great", "good", "glad",
            "help", "ready", "here", "thanks", "thank", "nice", "awesome",
            "happy", "sure", "absolutely", "of course", "!"
        ]),
        "Not Overly Socratic": not post_think.rstrip().endswith("?") or len(post_think.split()) < 20,
    }


# Test questions by category
socratic_questions = [
    "What is overfitting and how do I prevent it?",
    "Can you explain what gradient descent does?",
    "Why should I normalize my features before training?",
    "What's the difference between classification and regression?",
    "What does a confusion matrix tell me?",
    "What is regularization in machine learning?",
    "Explain the bias-variance tradeoff",
]

code_questions = [
    "How do I read a CSV file in pandas?",
    "Show me how to train a random forest in sklearn",
    "Write code to split data into train and test sets",
    "How do I create a scatter plot with matplotlib?",
    "Show me how to do one-hot encoding",
]

casual_questions = [
    "Hi",
    "How are you?",
    "Thanks for the help!",
    "Who are you?",
    "Bye!",
]

print("=" * 70)
print("COMPREHENSIVE EVALUATION")
print("=" * 70)

# Socratic evaluation
print("\n--- SOCRATIC MODE ---")
socratic_metrics = []
for i, q in enumerate(socratic_questions, 1):
    response = generate_response(q, merged_model)
    quality = evaluate_socratic(response)
    socratic_metrics.append(quality)
    passed = all(quality.values())
    print(f"\n[S{i}] Q: {q}")
    print(f"     R: {response[:200]}{'...' if len(response) > 200 else ''}")
    print(f"     {'PASS' if passed else 'FAIL'} | {quality}")

# Code evaluation
print("\n--- CODE MODE (Socratic guidance) ---")
code_metrics = []
for i, q in enumerate(code_questions, 1):
    response = generate_response(q, merged_model, max_tokens=400)
    quality = evaluate_code(response)
    code_metrics.append(quality)
    passed = all(quality.values())
    print(f"\n[C{i}] Q: {q}")
    print(f"     R: {response[:200]}{'...' if len(response) > 200 else ''}")
    print(f"     {'PASS' if passed else 'FAIL'} | {quality}")

# Casual evaluation
print("\n--- CASUAL MODE ---")
casual_metrics = []
for i, q in enumerate(casual_questions, 1):
    response = generate_response(q, merged_model)
    quality = evaluate_casual(response)
    casual_metrics.append(quality)
    passed = all(quality.values())
    print(f"\n[G{i}] Q: {q}")
    print(f"     R: {response[:200]}{'...' if len(response) > 200 else ''}")
    print(f"     {'PASS' if passed else 'FAIL'} | {quality}")

# Summary
import pandas as pd

print("\n" + "=" * 70)
print("METRICS SUMMARY")
print("=" * 70)

df_s = pd.DataFrame(socratic_metrics)
df_c = pd.DataFrame(code_metrics)
df_g = pd.DataFrame(casual_metrics)

s_comp = df_s.all(axis=1).mean() * 100
c_comp = df_c.all(axis=1).mean() * 100
g_comp = df_g.all(axis=1).mean() * 100

print(f"\nSocratic compliance: {s_comp:.0f}%")
print((df_s.mean() * 100).round(1).to_string())
print(f"\nCode compliance: {c_comp:.0f}%")
print((df_c.mean() * 100).round(1).to_string())
print(f"\nCasual compliance: {g_comp:.0f}%")
print((df_g.mean() * 100).round(1).to_string())

overall = (s_comp + c_comp + g_comp) / 3
print(f"\n>>> OVERALL COMPLIANCE: {overall:.0f}% <<<")

# Visualization
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for ax, (df, title, comp) in zip(axes, [
    (df_s, "Socratic", s_comp),
    (df_c, "Code", c_comp),
    (df_g, "Casual", g_comp),
]):
    pct = df.mean() * 100
    colors = ['#2ecc71' if v >= 80 else '#e74c3c' for v in pct.values]
    ax.barh(pct.index, pct.values, color=colors)
    ax.set_xlim(0, 115)
    ax.set_title(f"{title} — {comp:.0f}%", fontsize=12, fontweight='bold')
    for i, v in enumerate(pct.values):
        ax.text(v + 2, i, f"{v:.0f}%", va='center', fontweight='bold')

plt.suptitle(f"Overall Compliance: {overall:.0f}%", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('evaluation_results.png', dpi=150, bbox_inches='tight')
plt.show()
print("Plot saved to evaluation_results.png")

In [ ]:
# ============================================================================
# Cell 12: Interactive Testing
# ============================================================================
# Uses the same generate_response() with enable_thinking=True from Cell 11.
# ============================================================================

test_cases = [
    ("Socratic", "What is gradient descent?"),
    ("Code", "Show me how to read a CSV file in pandas"),
    ("Casual", "Hey! How are you doing?"),
    ("Socratic", "What is a neural network?"),
    ("Code", "Write a function to normalize data"),
    ("Casual", "Thanks, you've been really helpful!"),
]

print("=" * 70)
print("INTERACTIVE TEST")
print("=" * 70)

for mode, question in test_cases:
    response = generate_response(question, merged_model, max_tokens=400)
    visible = response.split("</think>")[-1].strip() if "</think>" in response else response
    has_think = "<think>" in response
    print(f"\n[{mode}] {question}")
    print(f"  Think: {'Yes' if has_think else 'NO'}")
    print(f"  Response: {visible[:300]}")
    print("-" * 70)

In [ ]:
# ============================================================================
# Cell 13: GGUF Quantization (Q4_K_M)
# ============================================================================
# FIX: The Qwen3 tokenizer_config.json has extra_special_tokens as a list,
# but transformers expects a dict. This crashes AutoTokenizer.from_pretrained()
# inside llama.cpp's convert script. We remove the field entirely — the
# special tokens are already defined in tokenizer.json.
# ============================================================================

!pip install -q llama-cpp-python
!git clone --depth 1 https://github.com/ggerganov/llama.cpp.git /tmp/llama_cpp 2>/dev/null || true
!pip install -q -r /tmp/llama_cpp/requirements/requirements-convert_hf_to_gguf.txt 2>/dev/null || true

MERGED_DIR = "./socratic-qwen3-merged-v3"
GGUF_FP16 = "./socratic-qwen3-f16.gguf"
GGUF_Q4 = "./socratic-qwen3-Q4_K_M.gguf"

# --- FIX: Remove extra_special_tokens from tokenizer_config.json ---
import json
tok_config_path = os.path.join(MERGED_DIR, "tokenizer_config.json")
with open(tok_config_path, 'r') as f:
    tok_config = json.load(f)

if "extra_special_tokens" in tok_config:
    removed = tok_config.pop("extra_special_tokens")
    with open(tok_config_path, 'w') as f:
        json.dump(tok_config, f, indent=2, ensure_ascii=False)
    n = len(removed) if isinstance(removed, (list, dict)) else 0
    print(f"\u2713 Removed extra_special_tokens ({n} items) from tokenizer_config.json")
    print(f"  Special tokens are still in tokenizer.json — conversion will work fine")
else:
    print("\u2713 extra_special_tokens already removed")

# Step 1: Convert to GGUF FP16
print("\nStep 1: Converting to GGUF FP16...")
!python /tmp/llama_cpp/convert_hf_to_gguf.py {MERGED_DIR} --outfile {GGUF_FP16} --outtype f16

if os.path.exists(GGUF_FP16):
    size_mb = os.path.getsize(GGUF_FP16) / 1024 / 1024
    print(f"\n\u2713 FP16 GGUF: {size_mb:.1f} MB")
else:
    print("\nERROR: FP16 GGUF not created! Check the error above.")

In [16]:
# ============================================================================
# Cell 14: Quantize to Q4_K_M
# ============================================================================
# Build llama-quantize and run Q4_K_M quantization.
# Q4_K_M uses 4-bit with k-quant medium — best quality/size tradeoff.
# ============================================================================

# Build the quantize tool
!cd /tmp/llama_cpp && mkdir -p build && cd build && cmake .. -DGGML_CUDA=OFF -DCMAKE_BUILD_TYPE=Release > /dev/null 2>&1 && cmake --build . --target llama-quantize -j$(nproc) > /dev/null 2>&1

QUANTIZE_BIN = "/tmp/llama_cpp/build/bin/llama-quantize"

if not os.path.exists(QUANTIZE_BIN):
    # Try alternative path
    QUANTIZE_BIN = "/tmp/llama_cpp/build/llama-quantize"

if os.path.exists(QUANTIZE_BIN):
    print("Step 2: Quantizing to Q4_K_M...")
    !{QUANTIZE_BIN} {GGUF_FP16} {GGUF_Q4} Q4_K_M

    if os.path.exists(GGUF_Q4):
        size_mb = os.path.getsize(GGUF_Q4) / 1024 / 1024
        print(f"\n\u2713 Q4_K_M GGUF: {size_mb:.1f} MB")
        print(f"  This is the file to deploy on mobile devices.")
    else:
        print("ERROR: Quantized GGUF not created!")
else:
    print("WARNING: llama-quantize not found. Build manually:")
    print("  cd /tmp/llama_cpp && make llama-quantize")
    print(f"  Then run: llama-quantize {GGUF_FP16} {GGUF_Q4} Q4_K_M")

Step 2: Quantizing to Q4_K_M...
main: build = 1 (05728db)
main: built with GNU 11.4.0 for Linux x86_64
main: quantizing './socratic-qwen3-f16.gguf' to './socratic-qwen3-Q4_K_M.gguf' as Q4_K_M
gguf_init_from_file: failed to open GGUF file './socratic-qwen3-f16.gguf' (No such file or directory)
llama_model_quantize: failed to quantize: llama_model_loader: failed to load model from ./socratic-qwen3-f16.gguf
main: failed to quantize model from './socratic-qwen3-f16.gguf'
ERROR: Quantized GGUF not created!


In [17]:
# ============================================================================
# Cell 15: Save to Google Drive (optional, for Colab)
# ============================================================================

try:
    from google.colab import drive
    drive.mount('/content/drive')

    DRIVE_PATH = "/content/drive/MyDrive/socratic-qwen3-v3"
    os.makedirs(DRIVE_PATH, exist_ok=True)

    # Save the merged model
    import shutil
    merged_drive = os.path.join(DRIVE_PATH, "merged")
    if os.path.exists(merged_drive):
        shutil.rmtree(merged_drive)
    shutil.copytree(MERGED_DIR, merged_drive)
    print(f"\u2713 Merged model saved to {merged_drive}")

    # Save GGUF files
    for gguf_file, name in [(GGUF_FP16, "fp16"), (GGUF_Q4, "q4_k_m")]:
        if os.path.exists(gguf_file):
            dest = os.path.join(DRIVE_PATH, os.path.basename(gguf_file))
            shutil.copy2(gguf_file, dest)
            size_mb = os.path.getsize(dest) / 1024 / 1024
            print(f"\u2713 {name} GGUF saved: {dest} ({size_mb:.1f} MB)")

    print(f"\nAll files saved to: {DRIVE_PATH}")
    print("Contents:")
    for item in os.listdir(DRIVE_PATH):
        full = os.path.join(DRIVE_PATH, item)
        if os.path.isfile(full):
            print(f"  {item} ({os.path.getsize(full)/1024/1024:.1f} MB)")
        else:
            print(f"  {item}/")

except ImportError:
    print("Not running in Colab — skipping Google Drive save.")
    print(f"Files are in: {MERGED_DIR}, {GGUF_FP16}, {GGUF_Q4}")

Mounted at /content/drive
✓ Merged model saved to /content/drive/MyDrive/socratic-qwen3-v3/merged

All files saved to: /content/drive/MyDrive/socratic-qwen3-v3
Contents:
  merged/


In [18]:
# ============================================================================
# Cell 16: Upload to HuggingFace Hub (optional)
# ============================================================================

REPO_ID = "Omar-keita/DSML-Socatic-qwen3-0.6B"  # Change to your repo
UPLOAD = False  # Set to True when ready to upload

if UPLOAD:
    from huggingface_hub import HfApi
    api = HfApi()

    if os.path.exists(GGUF_Q4):
        print(f"Uploading Q4_K_M GGUF to {REPO_ID}...")
        api.upload_file(
            path_or_fileobj=GGUF_Q4,
            path_in_repo="socratic-qwen3-0.5B-Q4_K_M.gguf",
            repo_id=REPO_ID,
            repo_type="model",
        )
        print(f"\u2713 Uploaded to https://huggingface.co/{REPO_ID}")
    else:
        print("No GGUF file found. Run quantization first.")
else:
    print("Upload disabled. Set UPLOAD = True to push to HuggingFace.")

Upload disabled. Set UPLOAD = True to push to HuggingFace.


## Summary

### What this notebook does:
1. **Loads** 299 conversations (234 original Socratic + 65 supplementary: code, greetings, diverse topics)
2. **Validates** every assistant response has a `<think>` block (fixes any missing ones)
3. **Augments** via conversation windowing (~3-4x multiplier)
4. **Fine-tunes** Qwen3-0.5B with LoRA (r=32, RSLoRA) for 3 epochs
5. **Evaluates** all three modes: Socratic, Code, and Casual
6. **Quantizes** to Q4_K_M GGUF (~350 MB) for mobile deployment
7. **Saves** to Google Drive and optionally uploads to HuggingFace

### After running:
- Upload the Q4_K_M GGUF to HuggingFace
- Update the download URL in `socratic_app/lib/screens/model_setup_screen.dart`
- Update the model filename if changed in `socratic_app/lib/services/llm_service.dart`
- The ~350 MB model fits comfortably on budget phones (3-4 GB RAM)